In [1]:
import torch
import torch.nn.functional as F

In [2]:
words = open('names.txt', 'r').read().splitlines()
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'mia',
 'charlotte',
 'amelia',
 'harper',
 'evelyn']

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {ch:i+1 for i,ch in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(stoi)

print(stoi)

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}


In [4]:
block_size = 4
X, Y = [], []

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [5]:
n_embd = 10
n_hidden = 100

C = torch.randn((vocab_size, n_embd))
W1 = torch.randn((n_embd * block_size, n_hidden))
b1 = torch.zeros(n_hidden)

W2 = torch.randn((n_hidden, vocab_size))
b2 = torch.zeros(vocab_size)

parameters = [C, W1, b1, W2, b2]

# normalize weights (important)
for p in parameters:
    p.data *= 0.1
    p.requires_grad = True


In [6]:
for step in range(15000):
    
    ix = torch.randint(0, X.shape[0], (64,))

    emb = C[X[ix]]
    h = emb.view(emb.shape[0], -1) @ W1 + b1
    h = torch.tanh(h)
    logits = h @ W2 + b2

    loss = F.cross_entropy(logits, Y[ix])

    for p in parameters:
        p.grad = None
    loss.backward()

    for p in parameters:
        p.data += -0.01 * p.grad

    if step % 500 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")

# =============================

Step 0, Loss: 3.2808
Step 500, Loss: 2.8360
Step 1000, Loss: 2.8629
Step 1500, Loss: 2.5454
Step 2000, Loss: 2.6880
Step 2500, Loss: 2.4539
Step 3000, Loss: 2.3840
Step 3500, Loss: 2.3049
Step 4000, Loss: 2.3901
Step 4500, Loss: 2.4064
Step 5000, Loss: 2.5052
Step 5500, Loss: 2.3630
Step 6000, Loss: 2.3776
Step 6500, Loss: 2.3010
Step 7000, Loss: 2.2584
Step 7500, Loss: 2.2052
Step 8000, Loss: 2.0289
Step 8500, Loss: 1.8558
Step 9000, Loss: 2.4889
Step 9500, Loss: 2.1263
Step 10000, Loss: 2.0481
Step 10500, Loss: 1.9966
Step 11000, Loss: 2.2105
Step 11500, Loss: 1.8575
Step 12000, Loss: 2.0440
Step 12500, Loss: 1.9638
Step 13000, Loss: 2.1006
Step 13500, Loss: 1.9986
Step 14000, Loss: 1.8663
Step 14500, Loss: 1.9268


In [7]:
def generate_from_input(start_str, num_samples=5):
    for _ in range(num_samples):
        out = list(start_str)

        # convert input → numbers
        context = [0] * block_size
        for ch in start_str:
            context = context[1:] + [stoi[ch]]

        for _ in range(20):
            emb = C[torch.tensor([context])]
            h = emb.view(1, -1) @ W1 + b1
            h = torch.tanh(h)
            logits = h @ W2 + b2

            probs = F.softmax(logits / 0.8, dim=1)

            ix = torch.multinomial(probs, num_samples=1).item()
            context = context[1:] + [ix]

            if ix == 0:
                break

            out.append(itos[ix])

        print(''.join(out))

In [15]:
generate_from_input("padmes")

padmesin
padmesaree
padmesh
padmesane
padmesha
